In [1]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

sys.path.append('../..')

In [3]:
import numpy as np

from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.monitor import Monitor

In [4]:
from vimms.Common import POSITIVE
from vimms.ChemicalSamplers import UniformRTAndIntensitySampler, GaussianChromatogramSampler, UniformMZFormulaSampler

from vimms_gym.env import DDAEnv
from vimms_gym.chemicals import generate_chemicals
from vimms_gym.evaluation import run_method

# 1. Parameters

In [5]:

# n_chemicals = (200, 500)
# mz_range = (100, 600)
# rt_range = (0, 300)
# intensity_range = (1E5, 1E10)


In [6]:
n_chemicals = (2000, 5000)
mz_range = (100, 600)
rt_range = (200, 1000)
intensity_range = (1E4, 1E10)

In [7]:
min_mz = mz_range[0]
max_mz = mz_range[1]
min_rt = rt_range[0]
max_rt = rt_range[1]
min_log_intensity = np.log(intensity_range[0])
max_log_intensity = np.log(intensity_range[1])

In [8]:
isolation_window = 0.7
N = 10
rt_tol = 15
mz_tol = 10
min_ms1_intensity = 5000
ionisation_mode = POSITIVE

enable_spike_noise = True
noise_density = 0.1
noise_max_val = 1E3

In [9]:
mz_sampler = UniformMZFormulaSampler(min_mz=min_mz, max_mz=max_mz)
ri_sampler = UniformRTAndIntensitySampler(min_rt=min_rt, max_rt=max_rt,
                                          min_log_intensity=min_log_intensity,
                                          max_log_intensity=max_log_intensity)
cr_sampler = GaussianChromatogramSampler()
samplers = {
    'mz': mz_sampler,
    'rt_intensity': ri_sampler,
    'chromatogram': cr_sampler
}

In [10]:
params = {
    'chemical_creator': {
        'mz_range': mz_range,
        'rt_range': rt_range,
        'intensity_range': intensity_range,
        'n_chemicals': n_chemicals,
        'mz_sampler': mz_sampler,
        'ri_sampler': ri_sampler,
        'cr_sampler': cr_sampler,
    },
    'noise': {
        'enable_spike_noise': enable_spike_noise,
        'noise_density': noise_density,
        'noise_max_val': noise_max_val,
        'mz_range': mz_range
    },
    'env': {
        'ionisation_mode': ionisation_mode,
        'rt_range': rt_range,
        'isolation_window': isolation_window,
        'mz_tol': mz_tol,
        'rt_tol': rt_tol,
    }
}

In [11]:
max_peaks = 100
in_dir = 'results_%d_%d' % (max_peaks, rt_tol)

In [12]:
total_timesteps = 20E6
n_eval_episodes = 10
deterministic = True

# 2. Training

## Train PPO

In [13]:
model_name = 'PPO'

In [14]:
# default parameters
learning_rate = 0.0003
batch_size = 64
n_steps = 2048
ent_coef = 0.0
gamma = 0.99
gae_lambda = 0.95

# modified parameters
# learning_rate = 0.0003
# batch_size = 64
# n_steps = 2048
# ent_coef = 0.01
# gamma = 0.80
# gae_lambda = 0.80

In [15]:
env = DDAEnv(max_peaks, params)
check_env(env)

### Multi-processing

In [16]:
import multiprocessing

def make_env(rank, seed=0):
    def _init():
        env = DDAEnv(max_peaks, params)
        env.seed(rank)
        env = Monitor(env)        
        return env
    set_random_seed(seed)
    return _init

num_cpu = multiprocessing.cpu_count()-1
num_cpu

191

In [17]:
if num_cpu > 100:
    num_cpu = 100

In [18]:
env = SubprocVecEnv([make_env(i) for i in range(num_cpu)]) 

In [19]:
tensorboard_log = './%s/%s_DDAEnv_tensorboard' % (in_dir, model_name)

model = PPO("MultiInputPolicy", env, 
            tensorboard_log=tensorboard_log, 
            learning_rate=learning_rate, batch_size=batch_size, n_steps=n_steps, 
            ent_coef=ent_coef, gamma=gamma, gae_lambda=gae_lambda)
model.learn(total_timesteps=total_timesteps)

In [20]:
# model = PPO("MultiInputPolicy", env, verbose=2, 
#             learning_rate=learning_rate, batch_size=batch_size, n_steps=n_steps, 
#             ent_coef=ent_coef, gamma=gamma, gae_lambda=gae_lambda)
# model.learn(total_timesteps=total_timesteps, log_interval=1)

In [21]:
fname = '%s/model_%s' % (in_dir, model_name)
model.save(fname)

In [22]:
mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=n_eval_episodes, 
                                          deterministic=deterministic)
mean_reward, std_reward

(727.2479392, 38.74288212977937)

# 2. Evaluation

Generate some chemical sets

In [23]:
n_eval_episodes = 1
eval_dir = 'evaluation'
methods = [
    'PPO',
    'TopN',
    'random',    
]

In [24]:
chemical_creator_params = params['chemical_creator']

chem_list = []
for i in range(n_eval_episodes):
    chems = generate_chemicals(chemical_creator_params)
    print(len(chems))
    chem_list.append(chems)

3577


Run different methods

In [25]:
max_peaks

100

In [26]:
out_dir = '%s/eval_%d_%d' % (eval_dir, max_peaks, rt_tol)
in_dir, out_dir

('results_100_15', 'evaluation/eval_100_15')

In [27]:
copy_params = dict(params)
copy_params['env']['rt_tol'] = rt_tol

for method in methods:
    banner = 'method = %s max_peaks = %d rt_tol = %d' % (method, max_peaks, rt_tol)
    print(banner)
    print()
    
    if method == 'PPO':
        fname = os.path.join(in_dir, 'model_%s.zip' % method)
        model = PPO.load(fname)
    elif method == 'DQN':
        fname = os.path.join(in_dir, 'model_%s.zip' % method)
        model = DQN.load(fname)
    else:
        model = None

    env = DDAEnv(max_peaks, copy_params)
    run_method(env, chem_list, method, out_dir, min_ms1_intensity=min_ms1_intensity, model=model, N=10, print_eval=True)
    print()

method = PPO max_peaks = 100 rt_tol = 15

Episode 0 finished after 3482 timesteps with reward 711.6451030380495
{'coverage_prop': '0.521', 'intensity_prop': '0.327', 'ms1/ms2 ratio': '0.175', 'efficiency': '0.629'}

method = TopN max_peaks = 100 rt_tol = 15

Episode 0 finished after 3660 timesteps with reward 587.424634756066
{'coverage_prop': '0.414', 'intensity_prop': '0.340', 'ms1/ms2 ratio': '0.102', 'efficiency': '0.446'}

method = random max_peaks = 100 rt_tol = 15

Episode 0 finished after 3956 timesteps with reward 547.143288350592
{'coverage_prop': '0.429', 'intensity_prop': '0.291', 'ms1/ms2 ratio': '0.011', 'efficiency': '0.392'}

